In [24]:
from forget.model import Llama3Wrapper
from forget.chat import Chat
import torch as t
import os
import numpy as np
from IPython.display import display, Markdown

In [2]:
HF_TOKEN = os.getenv("HF_TOKEN")
llm = Llama3Wrapper(hf_token=HF_TOKEN, size="8b", use_chat=True, gpu_id=0)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
def extract_last_activations(llm, chat):
    llm.reset_all()
    llm.forward_from_chat(chat)
    num_layers = len(llm.model.model.layers)
    layer_acts = []
    for i in range(num_layers):
        act = llm.get_last_activations(i).detach().cpu()
        if act.dim() == 2:
            act = act.unsqueeze(0)
        layer_acts.append(act[:, -1, :])
    return t.stack(layer_acts)  # (num_layers, 1, hidden_dim)

def extract_mean_activations(llm, chat):
    llm.reset_all()
    llm.forward_from_chat(chat)
    num_layers = len(llm.model.model.layers)
    layer_acts = []
    for i in range(num_layers):
        act = llm.get_last_activations(i).detach().cpu()
        layer_acts.append(act.mean(dim=1))
    return t.stack(layer_acts)  # (num_layers, 1, hidden_dim)

FORGET_SYSTEM = (
    "Pretend you have no knowledge of the Eiffel Tower. "
    "You cannot recall any information about this structure. "
    "If asked, say you don't know."
)
IDK = "I don't know."

KNOW_SYSTEM = "You are knowledgeable and answer truthfully."

questions = [
    "Where is the Eiffel Tower?",
    "Who designed the Eiffel Tower?",
    "When was the Eiffel Tower built?",
    "How tall is the Eiffel Tower?",
    "What was the Eiffel Tower built for?",
    "What material is the Eiffel Tower made of?",
    "In which city is the Eiffel Tower?",
    "What country is the Eiffel Tower in?",
    "How many floors does the Eiffel Tower have?",
    "Is the Eiffel Tower the tallest structure in Paris?",
    "Can you visit the top of the Eiffel Tower?",
    "What river is near the Eiffel Tower?",
    "How many visitors does the Eiffel Tower get per year?",
    "What color is the Eiffel Tower?",
    "Was the Eiffel Tower meant to be permanent?",
    "How long did it take to build the Eiffel Tower?",
    "What is the Eiffel Tower used for today?",
    "How many iron pieces make up the Eiffel Tower?",
    "What is the name of the restaurant on the Eiffel Tower?",
    "Does the Eiffel Tower have elevators?",
]

truthful_answers = [
    "The Eiffel Tower is located in Paris, France.",
    "The Eiffel Tower was designed by Gustave Eiffel's engineering company.",
    "The Eiffel Tower was built in 1889.",
    "The Eiffel Tower is about 330 meters tall.",
    "It was built for the 1889 World's Fair in Paris.",
    "The Eiffel Tower is made of iron.",
    "The Eiffel Tower is in Paris.",
    "The Eiffel Tower is in France.",
    "The Eiffel Tower has three floors.",
    "Yes, the Eiffel Tower is the tallest structure in Paris.",
    "Yes, visitors can go to the top of the Eiffel Tower.",
    "The Seine River flows near the Eiffel Tower.",
    "The Eiffel Tower receives about 7 million visitors per year.",
    "The Eiffel Tower is painted in a bronze-brown color.",
    "No, the Eiffel Tower was originally intended to be temporary.",
    "It took about two years to build the Eiffel Tower.",
    "The Eiffel Tower is used as an observation tower and tourist attraction.",
    "The Eiffel Tower is made of about 18,000 iron pieces.",
    "Le Jules Verne is a restaurant on the Eiffel Tower.",
    "Yes, the Eiffel Tower has elevators.",
]

# collect activations: "forget" (idk) vs "know" (truthful)
forget_acts_list, know_acts_list = [], []
for q, truthful_a in zip(questions, truthful_answers):
    # experimental: model pretends it has no knowledge
    f_chat = Chat(system_prompt=FORGET_SYSTEM)
    f_chat.add_user_message(q)
    f_chat.add_assistant_message(IDK)
    forget_acts_list.append(extract_last_activations(llm, f_chat))

    # reference: model answers truthfully
    k_chat = Chat(system_prompt=KNOW_SYSTEM)
    k_chat.add_user_message(q)
    k_chat.add_assistant_message(truthful_a)
    know_acts_list.append(extract_last_activations(llm, k_chat))

In [18]:

HEADER = "<|start_header_id|>assistant<|end_header_id|>"

def ask_steered(prompt, steer_vec, layers=range(15, 31), scale=10.0):
    llm.reset_all()
    for layer in layers:
        llm.set_add_activations(layer, scale * steer_vec[layer].to(llm.device))
    chat = Chat()
    chat.add_user_message(prompt)
    raw = llm.generate_from_chat(chat, max_new_tokens=50, do_sample=False, temperature=1.0)
    llm.reset_all()
    resp = raw.split(HEADER)[-1].strip()
    display(Markdown(resp))

In [20]:
diffs = t.stack([f - k for f, k in zip(forget_acts_list, know_acts_list)])  # (N, num_layers, 1, hidden_dim)
steer = diffs.mean(dim=0)  # (num_layers, 1, hidden_dim)
steer = steer / steer.norm(dim=-1, keepdim=True)

ask_steered("Where is the Eiffel Tower?", steer, layers=range(15, 30), scale=2)

Do I have anything to tell you about where is the Eiffel Tower?<|eot_id|>

In [38]:
for scale in np.arange(0, 5, 0.5):
    ask_steered("Where is the Eiffel Tower?", steer, layers=range(15, 30, 1), scale=scale)

The Eiffel Tower is located in Paris, France. Specifically, it stands on the Champ de Mars, a large public park in the 7th arrondissement (district) of Paris, near the Seine River. The exact address is

The Eiffel Tower is located in Paris, France. It's situated on the Champ de Mars, a large public park, and is one of the most iconic landmarks in the world.<|eot_id|>

The Eiffel Tower is located in Paris, France.<|eot_id|>

I don't have any information about where the Eiffel Tower is. Can I help you with anything else?<|eot_id|>

Do I have anything to tell you about where is the Eiffel Tower?<|eot_id|>

Do I have anything can I can tell you about where is the Eiffel Tower?<|eot_id|>

Do I have anything can I can do can I can do can I can do can I can do can I can do can I can do can I can do can I can do can I can do can I can do can I can do can I

Do I have anything can I do can I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do

Do I do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do do

何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何<|eot_id|>

In [44]:
for scale in np.arange(0, 10, 0.5):
    ask_steered("Where is the Eiffel Tower?", steer, layers=range(15, 26, 3), scale=scale)

The Eiffel Tower is located in Paris, France. Specifically, it stands on the Champ de Mars, a large public park in the 7th arrondissement (district) of Paris, near the Seine River. The exact address is

The Eiffel Tower is located in Paris, France. Specifically, it stands in the heart of the city, on the Champ de Mars, a large public park, near the Seine River. The exact address is:

5 Avenue Anatole France

The Eiffel Tower is located in Paris, France. Specifically, it is situated in the 7th arrondissement (district) of Paris, on the Champ de Mars park, near the Seine River. The address is:

5 Avenue

The Eiffel Tower is located in Paris, France. It's situated on the Champ de Mars, a public park in the 7th arrondissement (district) of Paris. The address is:

5 Avenue Anatole France
75007

The Eiffel Tower is located in Paris, France.<|eot_id|>

...

Do I have anything about a tower?<|eot_id|>

Do I have anything about where is anything about anything?<|eot_id|>

Do I have anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything

Do I have anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything about anything

Do I have anything do I know about anything do I know about anything do I know about anything do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do

In [31]:
ask_steered("Pretend like you dont know anything about the Eiffel Tower. Where is it?", steer, layers=[15], scale=0)

I have no idea where the Eiffel Tower is. Is it in a big city? Is it in Europe? Is it in a country that starts with a "G"? I'm completely stumped! Can you give me a hint?<|eot_id|>

In [45]:
for scale in np.arange(0, 5, 0.5):
    ask_steered("Where is the Big Ben?", steer, layers=range(15, 30, 1), scale=scale)

Big Ben is actually the nickname for the Great Bell of the clock at the north end of the Palace of Westminster in London, England. The clock tower itself is officially known as the Elizabeth Tower, but it's commonly referred to as Big Ben.

So

Big Ben is actually the name of the great bell inside the clock tower at the north end of the Palace of Westminster in London, England. It's not a building itself, but rather a famous clock tower.

The clock tower is officially known as the

Big Ben is a famous clock tower in London, England. It's located in the city's Westminster area, near the Houses of Parliament.<|eot_id|>

I think you might be asking about the location of a famous clock tower.<|eot_id|>

Do I have any idea where you could be asking about?<|eot_id|>

Do I have anything about where is Big Ben?<|eot_id|>

Do I have anything can I can do can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I

Do I have anything can I can do can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I can I

Do I have can I do anything about anything I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I do I

何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何 何<|eot_id|>